# Analyse Exploratoire — Dataset BuzzFeed / FakeNewsNet
## Comparaison avec LIAR et caracterisation du Domain Shift

**Objectif** : Explorer les datasets externes utilises pour l'evaluation out-of-domain, comprendre les differences avec LIAR, et preparer les donnees pour l'evaluation hors domaine.

**Datasets explores** :
1. **BuzzFeed Facebook Fact-Check** (2016) — 2 282 posts Facebook annotes par des journalistes
2. **FakeNewsNet PolitiFact** — 432 articles fake + 120 articles real avec titres

**Plan** :
1. Chargement et structure des datasets
2. Distribution des labels et mapping binaire
3. Analyse textuelle (longueur, style, vocabulaire)
4. Comparaison avec LIAR (domain shift)
5. Engagement sur les reseaux sociaux (BuzzFeed)
6. Preprocessing et sauvegarde pour l'evaluation OOD

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from collections import Counter
import re
import warnings
warnings.filterwarnings("ignore")

# Chemins
BUZZFEED_PATH = Path("../data/brutes/2016-10-facebook-fact-check/data/facebook-fact-check.csv")
FNN_DIR = Path("../data/brutes/FakeNewsNet/dataset")
LIAR_TRAITEES = Path("../data/Traitees")
DATA_OUT = Path("../data/Traitees")
DATA_OUT.mkdir(parents=True, exist_ok=True)

COLORS_BIN = ["#e74c3c", "#2ecc71"]
COLORS_4 = ["#e74c3c", "#e67e22", "#3498db", "#2ecc71"]

print("Imports OK")

Imports OK


## 1. Chargement des datasets externes

### 1.1 BuzzFeed Facebook Fact-Check (2016)

Ce dataset contient 2 282 posts Facebook publies par 9 pages d'information americaines pendant la campagne presidentielle 2016. Chaque post est annote avec un `Rating` parmi 4 niveaux : `mostly true`, `mixture of true and false`, `mostly false`, `no factual content`.

**Particularite** : Le dataset ne contient pas le texte des publications, seulement les URLs et metadonnees (engagement, type de post, page source). Pour l'evaluation textuelle, on utilisera le nom de la `Page` comme proxy categoriel.

In [2]:
# Chargement BuzzFeed
bf = pd.read_csv(BUZZFEED_PATH)

print(f"BuzzFeed Facebook Fact-Check : {len(bf):,} posts")
print(f"Colonnes : {list(bf.columns)}")
print(f"\nTypes de posts :")
print(bf["Post Type"].value_counts())
print(f"\nPages sources :")
print(bf["Page"].value_counts())
print(f"\nValeurs manquantes :")
print(bf.isnull().sum()[bf.isnull().sum() > 0])
bf.head(3)

BuzzFeed Facebook Fact-Check : 2,282 posts
Colonnes : ['account_id', 'post_id', 'Category', 'Page', 'Post URL', 'Date Published', 'Post Type', 'Rating', 'Debate', 'share_count', 'reaction_count', 'comment_count']

Types de posts :
Post Type
link     1780
video     291
photo     207
text        4
Name: count, dtype: int64

Pages sources :
Page
Politico             536
CNN Politics         409
Eagle Rising         286
Right Wing News      268
Occupy Democrats     209
ABC News Politics    200
Addicting Info       140
The Other 98%        122
Freedom Daily        112
Name: count, dtype: int64

Valeurs manquantes :
Debate            1984
share_count         70
reaction_count       2
comment_count        2
dtype: int64


,account_id,post_id,Category,Page,Post URL,Date Published,Post Type,Rating,Debate,share_count,reaction_count,comment_count
0,184096565021911,1035057923259100,mainstream,ABC News Politics,https://www.facebook.com/ABCNewsPolitics/posts...,2016-09-19,video,no factual content,NaN,NaN,146.0,15.0
1,184096565021911,1035269309904628,mainstream,ABC News Politics,https://www.facebook.com/ABCNewsPolitics/posts...,2016-09-19,link,mostly true,NaN,1.0,33.0,34.0
2,184096565021911,1035305953234297,mainstream,ABC News Politics,https://www.facebook.com/ABCNewsPolitics/posts...,2016-09-19,link,mostly true,NaN,34.0,63.0,27.0


### 1.2 FakeNewsNet — PolitiFact

FakeNewsNet fournit des articles de presse lies a des fact-checks PolitiFact. Chaque article a un `title` (le texte qu'on utilisera pour l'evaluation OOD) et un label binaire implicite (le fichier source : `politifact_fake.csv` ou `politifact_real.csv`).

In [3]:
# Chargement FakeNewsNet PolitiFact
pf_fake = pd.read_csv(FNN_DIR / "politifact_fake.csv")
pf_real = pd.read_csv(FNN_DIR / "politifact_real.csv")

pf_fake["label_binary"] = 0
pf_fake["label_name"] = "Fake"
pf_real["label_binary"] = 1
pf_real["label_name"] = "Real"

fnn = pd.concat([pf_fake, pf_real], ignore_index=True)

# Supprimer les lignes sans titre
fnn = fnn.dropna(subset=["title"]).reset_index(drop=True)

print(f"FakeNewsNet PolitiFact : {len(fnn):,} articles")
print(f"  Fake : {len(pf_fake):,}")
print(f"  Real : {len(pf_real):,}")
print(f"  Apres nettoyage (titres non vides) : {len(fnn):,}")
print(f"\nColonnes : {list(fnn.columns)}")
print(f"\nExemples de titres Fake :")
for t in fnn[fnn["label_binary"] == 0]["title"].head(3):
    print(f"  - {t[:100]}")
print(f"\nExemples de titres Real :")
for t in fnn[fnn["label_binary"] == 1]["title"].head(3):
    print(f"  - {t[:100]}")

FakeNewsNet PolitiFact : 1,056 articles
  Fake : 432
  Real : 624
  Apres nettoyage (titres non vides) : 1,056

Colonnes : ['id', 'news_url', 'title', 'tweet_ids', 'label_binary', 'label_name']

Exemples de titres Fake :
  - BREAKING: First NFL Team Declares Bankruptcy Over Kneeling Thugs
  - Court Orders Obama To Pay $400 Million In Restitution
  - UPDATE: Second Roy Moore Accuser Works For Michelle Obama Right NOW -

Exemples de titres Real :
  - National Federation of Independent Business
  - comments in Fayetteville NC
  - Romney makes pitch, hoping to close deal : Elections : The Rocky Mountain News


## 2. Distribution des labels

### 2.1 BuzzFeed — 4 niveaux de veracite + mapping binaire

Le BuzzFeed dataset utilise 4 niveaux :
- `mostly true` — plutot vrai
- `mixture of true and false` — melange
- `mostly false` — plutot faux
- `no factual content` — pas de contenu factuel

**Mapping binaire** :
- **Fake (0)** : `mostly false` + `no factual content`
- **Real (1)** : `mostly true` + `mixture of true and false`

Ce mapping est coherent avec celui du LIAR (les categories ambigues comme `mixture` sont classees Real, comme `half-true` dans LIAR).

In [4]:
# Distribution des 4 ratings BuzzFeed
rating_order = ["mostly false", "no factual content", "mixture of true and false", "mostly true"]
rating_counts = bf["Rating"].value_counts().reindex(rating_order)

fig = px.bar(
    x=rating_order, y=rating_counts.values,
    color=rating_order,
    color_discrete_sequence=COLORS_4,
    title="BuzzFeed — Distribution des 4 niveaux de veracite",
    labels={"x": "Rating", "y": "Nombre de posts"},
    text=rating_counts.values,
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, template="plotly_white", height=450)
fig.show()

# Mapping binaire BuzzFeed
BINARY_MAP_BF = {
    "mostly false": 0,
    "no factual content": 0,
    "mixture of true and false": 1,
    "mostly true": 1,
}
bf["label_binary"] = bf["Rating"].map(BINARY_MAP_BF)
bf["label_name"] = bf["label_binary"].map({0: "Fake", 1: "Real"})

bin_counts_bf = bf["label_name"].value_counts()
print(f"\nBuzzFeed — Distribution binaire :")
print(f"  Real : {bin_counts_bf.get('Real', 0):,} ({bin_counts_bf.get('Real', 0)/len(bf)*100:.1f}%)")
print(f"  Fake : {bin_counts_bf.get('Fake', 0):,} ({bin_counts_bf.get('Fake', 0)/len(bf)*100:.1f}%)")


BuzzFeed — Distribution binaire :
  Real : 1,914 (83.9%)
  Fake : 368 (16.1%)


In [5]:
# Distribution binaire FakeNewsNet
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["BuzzFeed — Fake vs Real", "FakeNewsNet PolitiFact — Fake vs Real"],
    specs=[[{"type": "pie"}, {"type": "pie"}]],
)

# BuzzFeed
fig.add_trace(go.Pie(
    labels=["Fake", "Real"],
    values=[bin_counts_bf.get("Fake", 0), bin_counts_bf.get("Real", 0)],
    marker_colors=COLORS_BIN,
    textinfo="label+percent+value",
    hole=0.4,
), row=1, col=1)

# FakeNewsNet
fnn_counts = fnn["label_name"].value_counts()
fig.add_trace(go.Pie(
    labels=["Fake", "Real"],
    values=[fnn_counts.get("Fake", 0), fnn_counts.get("Real", 0)],
    marker_colors=COLORS_BIN,
    textinfo="label+percent+value",
    hole=0.4,
), row=1, col=2)

fig.update_layout(title="Distribution binaire des datasets externes", template="plotly_white", height=400)
fig.show()

print(f"FakeNewsNet — Fake: {fnn_counts.get('Fake', 0)} ({fnn_counts.get('Fake', 0)/len(fnn)*100:.1f}%)")
print(f"FakeNewsNet — Real: {fnn_counts.get('Real', 0)} ({fnn_counts.get('Real', 0)/len(fnn)*100:.1f}%)")

FakeNewsNet — Fake: 432 (40.9%)
FakeNewsNet — Real: 624 (59.1%)


## 3. Analyse textuelle — FakeNewsNet

Le dataset FakeNewsNet contient des `title` d'articles qu'on peut analyser textuellement et comparer avec les `statement` du LIAR.

In [6]:
# Longueur des titres FakeNewsNet
fnn["n_words"] = fnn["title"].str.split().str.len()
fnn["n_chars"] = fnn["title"].str.len()

print("=== Statistiques textuelles FakeNewsNet ===")
print(fnn[["n_words", "n_chars"]].describe().round(1))

# Charger LIAR pour comparaison
liar = pd.read_parquet(LIAR_TRAITEES / "liar_complet.parquet")
liar_words = liar["n_words"]

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"LIAR — Longueur des statements (n={len(liar):,})",
    f"FakeNewsNet — Longueur des titres (n={len(fnn):,})"
])

fig.add_trace(go.Histogram(
    x=liar_words, nbinsx=50, marker_color="#3498db",
    name="LIAR", opacity=0.8
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=fnn["n_words"], nbinsx=50, marker_color="#e67e22",
    name="FakeNewsNet", opacity=0.8
), row=1, col=2)

fig.update_xaxes(title_text="Nombre de mots", row=1, col=1)
fig.update_xaxes(title_text="Nombre de mots", row=1, col=2)
fig.update_layout(
    title="Comparaison de la longueur des textes : LIAR vs FakeNewsNet",
    template="plotly_white", height=400, showlegend=False
)
fig.show()

print(f"\nLIAR     — Mediane : {liar_words.median():.0f} mots, Moyenne : {liar_words.mean():.1f}")
print(f"FakeNews — Mediane : {fnn['n_words'].median():.0f} mots, Moyenne : {fnn['n_words'].mean():.1f}")

=== Statistiques textuelles FakeNewsNet ===
       n_words  n_chars
count   1056.0   1056.0
mean       9.5     59.9
std        5.2     31.3
min        1.0     10.0
25%        6.0     38.0
50%        9.0     59.0
75%       12.0     78.0
max       53.0    340.0



LIAR     — Mediane : 17 mots, Moyenne : 18.0
FakeNews — Mediane : 9 mots, Moyenne : 9.5


In [7]:
# Box plot longueur par label — FakeNewsNet
fig = px.box(
    fnn, x="label_name", y="n_words", color="label_name",
    color_discrete_map={"Fake": "#e74c3c", "Real": "#2ecc71"},
    title="FakeNewsNet — Nombre de mots par classe",
    labels={"n_words": "Nombre de mots", "label_name": "Classe"},
)
fig.update_layout(showlegend=False, template="plotly_white", height=400)
fig.show()

print(f"Fake — Mediane : {fnn[fnn['label_binary']==0]['n_words'].median():.0f} mots")
print(f"Real — Mediane : {fnn[fnn['label_binary']==1]['n_words'].median():.0f} mots")

Fake — Mediane : 11 mots
Real — Mediane : 7 mots


In [8]:
# Analyse du vocabulaire — comparaison LIAR vs FakeNewsNet
stopwords_basic = {"the", "a", "an", "is", "are", "was", "were", "of", "to", "in",
                   "for", "and", "on", "that", "it", "with", "as", "at", "by", "from",
                   "or", "be", "this", "has", "have", "had", "not", "but", "they",
                   "he", "she", "we", "his", "her", "its", "our", "their", "will",
                   "would", "than", "more", "been", "who", "which", "about", "said"}

def get_vocab(texts, n=30):
    words = " ".join(texts.astype(str)).lower().split()
    words = [w for w in words if w not in stopwords_basic and len(w) > 2]
    return Counter(words)

vocab_liar = get_vocab(liar["statement"])
vocab_fnn = get_vocab(fnn["title"])

# Top 20 mots les plus frequents
top_liar = vocab_liar.most_common(20)
top_fnn = vocab_fnn.most_common(20)

fig = make_subplots(rows=1, cols=2, subplot_titles=["LIAR — Top 20 mots", "FakeNewsNet — Top 20 mots"])

fig.add_trace(go.Bar(
    y=[w for w, c in reversed(top_liar)],
    x=[c for w, c in reversed(top_liar)],
    orientation="h", marker_color="#3498db", name="LIAR"
), row=1, col=1)

fig.add_trace(go.Bar(
    y=[w for w, c in reversed(top_fnn)],
    x=[c for w, c in reversed(top_fnn)],
    orientation="h", marker_color="#e67e22", name="FakeNewsNet"
), row=1, col=2)

fig.update_layout(
    title="Comparaison du vocabulaire : LIAR vs FakeNewsNet",
    template="plotly_white", height=550, showlegend=False
)
fig.show()

# Chevauchement de vocabulaire
set_liar = set(vocab_liar.keys())
set_fnn = set(vocab_fnn.keys())
overlap = set_liar & set_fnn

print(f"\nVocabulaire unique LIAR   : {len(set_liar):,} mots")
print(f"Vocabulaire unique FakeNewsNet : {len(set_fnn):,} mots")
print(f"Chevauchement            : {len(overlap):,} mots ({len(overlap)/len(set_fnn)*100:.1f}% du vocab FNN)")
print(f"Mots exclusifs FakeNewsNet : {len(set_fnn - set_liar):,} ({(len(set_fnn - set_liar))/len(set_fnn)*100:.1f}%)")


Vocabulaire unique LIAR   : 22,422 mots
Vocabulaire unique FakeNewsNet : 3,797 mots
Chevauchement            : 2,341 mots (61.7% du vocab FNN)
Mots exclusifs FakeNewsNet : 1,456 (38.3%)


## 4. Analyse BuzzFeed — Engagement et orientation politique

Le dataset BuzzFeed contient des metadonnees precieuses sur l'engagement (shares, reactions, commentaires) et la categorie de la page source (mainstream, left, right).

In [9]:
# Rating par categorie de page (mainstream / left / right)
cat_rating = bf.groupby(["Category", "Rating"]).size().reset_index(name="count")

fig = px.bar(
    cat_rating, x="Rating", y="count", color="Category",
    barmode="group",
    category_orders={"Rating": rating_order, "Category": ["mainstream", "left", "right"]},
    color_discrete_map={"mainstream": "#3498db", "left": "#2ecc71", "right": "#e74c3c"},
    title="BuzzFeed — Distribution des ratings par orientation de la page",
    labels={"count": "Nombre de posts"},
)
fig.update_layout(template="plotly_white", height=450)
fig.show()

# Taux de Fake par categorie
for cat in ["mainstream", "left", "right"]:
    sub = bf[bf["Category"] == cat]
    fake_rate = (sub["label_binary"] == 0).mean() * 100
    print(f"{cat:12s} — Taux Fake : {fake_rate:.1f}% (n={len(sub)})")

mainstream   — Taux Fake : 4.5% (n=1145)
left         — Taux Fake : 29.3% (n=471)
right        — Taux Fake : 26.7% (n=666)


In [10]:
# Engagement moyen par rating
engagement_cols = ["share_count", "reaction_count", "comment_count"]
engagement_by_rating = bf.groupby("Rating")[engagement_cols].mean().reindex(rating_order)

fig = px.bar(
    engagement_by_rating.reset_index().melt(id_vars="Rating", var_name="Metric", value_name="Moyenne"),
    x="Rating", y="Moyenne", color="Metric",
    barmode="group",
    category_orders={"Rating": rating_order},
    title="BuzzFeed — Engagement moyen par rating",
    labels={"Moyenne": "Nombre moyen"},
    color_discrete_sequence=["#9b59b6", "#f39c12", "#1abc9c"],
)
fig.update_layout(template="plotly_white", height=450)
fig.show()

print("Engagement moyen par rating :")
print(engagement_by_rating.round(1))

Engagement moyen par rating :
                           share_count  reaction_count  comment_count
Rating                                                               
mostly false                    3570.3          5896.2          506.3
no factual content             19648.3         19518.2         1469.3
mixture of true and false       5083.9          6216.4          750.4
mostly true                     1676.0          2964.3          331.3


## 5. Caracterisation du Domain Shift — Tableau comparatif

Synthese des differences structurelles entre LIAR et les datasets externes.

In [11]:
# Tableau comparatif des datasets
comparison = pd.DataFrame({
    "Critere": [
        "Nombre d'echantillons",
        "Type de texte",
        "Longueur mediane (mots)",
        "Labels d'origine",
        "Equilibre binaire (% Fake)",
        "Source de verite",
        "Periode",
        "Vocabulaire specifique",
    ],
    "LIAR": [
        f"{len(liar):,}",
        "Declarations politiques courtes",
        f"{liar_words.median():.0f}",
        "6 niveaux (pants-fire → true)",
        f"{(liar['label_binary']==0).mean()*100:.1f}%",
        "PolitiFact (fact-checkers)",
        "2007-2016",
        "Politique US (noms de politiciens, bills, etats)",
    ],
    "FakeNewsNet": [
        f"{len(fnn):,}",
        "Titres d'articles de presse",
        f"{fnn['n_words'].median():.0f}",
        "Binaire (fake/real)",
        f"{(fnn['label_binary']==0).mean()*100:.1f}%",
        "PolitiFact (fact-checkers)",
        "2015-2018",
        "Medias, sensationnalisme, clickbait",
    ],
    "BuzzFeed": [
        f"{len(bf):,}",
        "Posts Facebook (URLs, pas de texte)",
        "N/A (pas de texte)",
        "4 niveaux (mostly false → mostly true)",
        f"{(bf['label_binary']==0).mean()*100:.1f}%",
        "BuzzFeed News (journalistes)",
        "Sept-Oct 2016",
        "N/A",
    ],
})

print("=== Comparaison des datasets ===")
comparison.set_index("Critere")

=== Comparaison des datasets ===


,LIAR,FakeNewsNet,BuzzFeed
Critere,,,
Nombre d'echantillons,"12,791","1,056","2,282"
Type de texte,Declarations politiques courtes,Titres d'articles de presse,"Posts Facebook (URLs, pas de texte)"
Longueur mediane (mots),17,9,N/A (pas de texte)
Labels d'origine,6 niveaux (pants-fire → true),Binaire (fake/real),4 niveaux (mostly false → mostly true)
Equilibre binaire (% Fake),44.2%,40.9%,16.1%
Source de verite,PolitiFact (fact-checkers),PolitiFact (fact-checkers),BuzzFeed News (journalistes)
Periode,2007-2016,2015-2018,Sept-Oct 2016
Vocabulaire specifique,"Politique US (noms de politiciens, bills, etats)","Medias, sensationnalisme, clickbait",N/A


## 6. Preprocessing et sauvegarde pour l'evaluation OOD

On prepare le dataset FakeNewsNet PolitiFact pour l'evaluation hors domaine :
- Nettoyage du texte (meme pipeline que LIAR)
- Sauvegarde en Parquet

**Note** : Le dataset BuzzFeed ne contient pas de texte exploitable pour un modele NLP textuel. Il sera utilise pour l'analyse de l'engagement et des biais par orientation politique, mais **FakeNewsNet PolitiFact sera le dataset principal pour l'evaluation out-of-domain textuelle**.

In [12]:
def clean_text(text: str) -> str:
    """Nettoyage basique du texte (meme pipeline que LIAR EDA)."""
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Preprocessing FakeNewsNet
fnn["title_clean"] = fnn["title"].apply(clean_text)

# Sauvegarder
fnn_out = fnn[["id", "title", "title_clean", "label_binary", "label_name", "n_words", "n_chars"]].copy()
fnn_out.to_parquet(DATA_OUT / "fakenewsnet_politifact.parquet", index=False)
print(f"FakeNewsNet sauvegarde : {len(fnn_out):,} articles -> {DATA_OUT / 'fakenewsnet_politifact.parquet'}")

# Sauvegarder aussi BuzzFeed (pour analyse engagement)
bf_out = bf[["post_id", "Category", "Page", "Post Type", "Rating", 
             "label_binary", "label_name",
             "share_count", "reaction_count", "comment_count"]].copy()
bf_out.to_parquet(DATA_OUT / "buzzfeed_facebook.parquet", index=False)
print(f"BuzzFeed sauvegarde    : {len(bf_out):,} posts -> {DATA_OUT / 'buzzfeed_facebook.parquet'}")

# Verification
print(f"\nExemples de titres nettoyes FakeNewsNet :")
for i, row in fnn_out.head(3).iterrows():
    print(f"  [{row['label_name']}] {row['title_clean'][:80]}...")

FakeNewsNet sauvegarde : 1,056 articles -> ..\data\Traitees\fakenewsnet_politifact.parquet
BuzzFeed sauvegarde    : 2,282 posts -> ..\data\Traitees\buzzfeed_facebook.parquet

Exemples de titres nettoyes FakeNewsNet :
  [Fake] breaking first nfl team declares bankruptcy over kneeling thugs...
  [Fake] court orders obama to pay million in restitution...
  [Fake] update second roy moore accuser works for michelle obama right now...


## 7. Synthese — Domain Shift identifie

### Differences structurelles (LIAR vs datasets externes)

| Dimension | LIAR | FakeNewsNet / BuzzFeed | Impact attendu |
|-----------|------|----------------------|----------------|
| **Longueur** | Textes courts (mediane ~17 mots) | Titres plus longs, articles complets | Mean pooling (GloVe/W2V) dilue le signal sur les textes longs |
| **Style** | Declarations directes de politiciens | Style journalistique, sensationnaliste, clickbait | Les n-grams appris sur LIAR ne se transferent pas |
| **Vocabulaire** | Politique US specifique | Medias, evenements divers, vocabulaire emotionnel | Mots hors-vocabulaire TF-IDF → features nulles |
| **Distribution** | ~44% Fake / 56% Real | FNN: ~78% Fake / 22% Real | Le modele voit une distribution tres differente |
| **Granularite** | 6 niveaux de veracite | Binaire (fake/real) | Le mapping binaire lisse les nuances |

### Implications pour l'evaluation out-of-domain

1. **TF-IDF** sera le plus impacte : vocabulaire specifique, n-grams appris sur les declarations politiques courtes
2. **GloVe/Word2Vec** devrait mieux transferer : les embeddings pre-entraines couvrent un vocabulaire plus large
3. **BERT/RoBERTa** devrait etre le plus robuste : les embeddings contextuels generalisent mieux le sens
4. **FakeNewsNet PolitiFact** est le candidat ideal pour l'evaluation OOD textuelle (il a des titres + des labels binaires)

### Attention au biais de desequilibre

FakeNewsNet est fortement desequilibre (~78% Fake). Un modele qui predit toujours "Fake" obtient 78% d'accuracy. Les metriques d'evaluation doivent inclure F1, precision et recall par classe.